# Validate Key Vault Connectivity

This notebook validates Key Vault access, discovers additional vaults in scope, and writes a connectivity report for downstream reporting.

In [ ]:
import json
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

from azure.identity import DefaultAzureCredential
from pyspark.sql import SparkSession
from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType

sys.path.insert(0, str(Path.cwd().parent / "src"))
from key_vault_connector import KeyVaultConnector

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

spark = SparkSession.builder.appName("PurviewKeyVaultValidation").getOrCreate()
root = Path.cwd().parent
raw_root = root / "data" / "raw"
processed_root = root / "data" / "processed"
reports_root = root / "data" / "reports"

for directory in (raw_root, processed_root, reports_root):
    directory.mkdir(parents=True, exist_ok=True)

setup_payload = {}
setup_path = raw_root / "setup_audit_config.json"
if setup_path.exists():
    with setup_path.open("r", encoding="utf-8") as handle:
        setup_payload = json.load(handle)

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID") or setup_payload.get("subscription_id")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
configured_vault = os.getenv("KEY_VAULT_NAME") or (setup_payload.get("key_vault") or {}).get("vault")
credential = DefaultAzureCredential()

print(json.dumps({
    "subscription_id_present": bool(subscription_id),
    "resource_group": resource_group,
    "configured_vault": configured_vault,
}, indent=2))

In [ ]:
discovered_vaults = []
if subscription_id:
    discovered_vaults = KeyVaultConnector.discover_key_vaults(subscription_id, resource_group=resource_group, credential=credential)

vault_names = []
for vault in discovered_vaults:
    name = vault.get("name")
    if name and name not in vault_names:
        vault_names.append(name)

if configured_vault and configured_vault not in vault_names:
    vault_names.insert(0, configured_vault)

print(f"Vaults queued for validation: {vault_names}")

In [ ]:
validation_records = []
for vault_name in vault_names:
    connector = KeyVaultConnector(vault_name, credential)
    access_ok = connector.check_key_vault_access()
    secrets = connector.get_vault_secrets_info() if access_ok else []
    connectivity = connector.check_fabric_connectivity()

    validation_records.append({
        "vault_name": vault_name,
        "accessible": access_ok,
        "secret_count": len(secrets),
        "fabric_connected": connectivity.get("is_connected", False),
        "status": connectivity.get("status", "UNKNOWN"),
        "workspace_configured": connectivity.get("workspace_configured", False),
        "candidate_secret_names": ",".join(connectivity.get("evidence", {}).get("candidate_secret_names", [])),
        "remediation_steps": " | ".join(connectivity.get("remediation_steps", [])),
        "validated_at": datetime.utcnow().isoformat(),
    })

schema = StructType([
    StructField("vault_name", StringType(), True),
    StructField("accessible", BooleanType(), True),
    StructField("secret_count", IntegerType(), True),
    StructField("fabric_connected", BooleanType(), True),
    StructField("status", StringType(), True),
    StructField("workspace_configured", BooleanType(), True),
    StructField("candidate_secret_names", StringType(), True),
    StructField("remediation_steps", StringType(), True),
    StructField("validated_at", StringType(), True),
])

validation_df = spark.createDataFrame(validation_records) if validation_records else spark.createDataFrame([], schema)
display(validation_df)

In [ ]:
validation_path = processed_root / "key_vault_validation"
validation_df.write.mode("overwrite").parquet(str(validation_path))

report = {
    "generated_at": datetime.utcnow().isoformat(),
    "vaults": validation_records,
    "connected_count": len([record for record in validation_records if record.get("fabric_connected")]),
    "accessible_count": len([record for record in validation_records if record.get("accessible")]),
}

report_path = reports_root / "key_vault_connectivity.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

print(f"Saved validation parquet to {validation_path}")
print(f"Saved validation report to {report_path}")
print(json.dumps(report, indent=2))